### Links and Resources
| Function | Use |
|---|---|
| [`when` / `otherwise`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.when.html) | The PySpark CASE expression |
| [`coalesce`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.coalesce.html) | First non-null |
| [`nullif`, `nvl`, `nvl2`, `ifnull`](https://spark.apache.org/docs/latest/api/sql/index.html#nullif) | NULL handling |
| [SQL `CASE`](https://spark.apache.org/docs/latest/sql-ref-syntax-qry-select-case.html) | The SQL form |

In [ ]:
import os
if not os.getenv("DATABRICKS_RUNTIME_VERSION"):
    from spark_setup import spark, display  # local run through Databricks Connect

from pyspark.sql import functions as F

# Conditional Functions

How you express *if/else* in a Spark expression. There's only one rule worth remembering: **NULL is not False.** Most bugs in this area come from forgetting that any comparison with NULL returns NULL, not False, and that `WHEN x = NULL` never matches.

In [ ]:
df = spark.createDataFrame([
    (1, "Alice",   850, "active"),
    (2, "Bob",     720, "active"),
    (3, "Charlie", None, "active"),
    (4, "Dana",    420, "churned"),
    (5, "Erin",    None, None),
], ["id", "name", "score", "status"])
display(df)

## `when` / `otherwise` — the workhorse

Chain `when` calls; the first match wins. The final `otherwise` is the default — omit it and unmatched rows become NULL.

In [ ]:
df.select(
    "name", "score",
    F.when(F.col("score") >= 800, "A")
     .when(F.col("score") >= 600, "B")
     .when(F.col("score") >= 400, "C")
     .otherwise("F")
     .alias("grade"),
).show()

## SQL `CASE` — same thing in `selectExpr`

Reads better when there are many branches.

In [ ]:
df.selectExpr("""
    name, score,
    CASE
        WHEN score >= 800 THEN 'A'
        WHEN score >= 600 THEN 'B'
        WHEN score >= 400 THEN 'C'
        ELSE 'F'
    END AS grade
""").show()

## `coalesce` — first non-null

Takes any number of columns, returns the first that isn't NULL. The canonical way to apply defaults.

In [ ]:
df.select(
    "name",
    F.coalesce("score", F.lit(0)).alias("score_default_0"),
    F.coalesce("status", F.lit("unknown")).alias("status_default"),
).show()

## `nullif`, `nvl`, `nvl2`, `ifnull`

Less famous but handy:

- `nullif(a, b)` — returns NULL if `a == b`, else `a`. Useful to convert sentinel values back to NULL.
- `nvl(a, b)` — alias of `coalesce(a, b)`.
- `nvl2(a, b, c)` — `b` if `a` is not NULL, else `c`.
- `ifnull(a, b)` — alias of `coalesce(a, b)`.

Most live as SQL functions; call them via `F.expr(...)`.

In [ ]:
df.selectExpr(
    "name", "score",
    "nullif(score, 0)         AS no_zero",
    "nvl(score, -1)            AS score_or_-1",
    "nvl2(score, 'has', 'no')  AS has_score",
).show()

## The NULL trap

`x = NULL` is *never* True. Use `IS NULL` / `isNull()` / `isNotNull()`.

In [ ]:
# Wrong — always returns nothing because score = NULL is NULL, not True
df.filter(F.col("score") == F.lit(None)).count()

# Right
print("nulls in score:", df.filter(F.col("score").isNull()).count())

## `when` returns a Column — combine freely

You can wrap `when` around any expression, including aggregates and other `when`s.

In [ ]:
df.groupBy("status").agg(
    F.count("*").alias("rows"),
    F.sum(F.when(F.col("score").isNotNull(), 1).otherwise(0)).alias("non_null_scores"),
    F.avg(F.when(F.col("score") >= 600, F.col("score"))).alias("avg_high_score"),
).show()

## Takeaways

- `when().when().otherwise()` is *the* if/else — first match wins.
- Use `coalesce` for defaults; it short-circuits and accepts N arguments.
- `x = NULL` is NULL, never True. Use `isNull()` / `isNotNull()`.
- Conditional aggregates: `sum(when(cond, 1).otherwise(0))` counts matches.